# Scirpy / MuData → tcri loader example

This notebook shows a practical way to load a `.h5mu` object, align GEX with AIRR clonotypes,
create a `tcri` covariate as a **tissue × site** combination, and register the object with
`tcri.ml.TCRIModel.setup_anndata`.

> Dataset used below: `/Users/ceglian/Data/gbm/scripy/gbm_gex_vdj.h5mu`

In [ ]:
import numpy as np
import pandas as pd
import muon as mu
import scirpy as ir
import tcri

## Built-in example datasets

`scirpy` ships useful built-in datasets for quick validation:

- `ir.datasets.wu2020()` (MuData)
- `ir.datasets.wu2020_3k()` (AnnData)
- `ir.datasets.maynard2020()` (AnnData)
- `ir.datasets.stephenson2021_5k()` (MuData)

`muon`/`mudata` do not ship a comparable built-in dataset module.

In [ ]:
dataset_funcs = [
    n for n in dir(ir.datasets)
    if not n.startswith("_") and callable(getattr(ir.datasets, n))
]
print([n for n in dataset_funcs if n in {
    "wu2020", "wu2020_3k", "maynard2020", "stephenson2021_5k"
}])

## Load the GBM MuData object

In [ ]:
dataset = "/Users/ceglian/Data/gbm/scripy/gbm_gex_vdj.h5mu"
mdata = mu.read_h5mu(dataset)

print("modalities:", list(mdata.mod.keys()))
print("global n_obs:", mdata.n_obs)
print("gex shape:", mdata.mod["gex"].shape)
print("airr shape:", mdata.mod["airr"].shape)

## Build a tcri-ready AnnData

Use `tcri.pp.from_mudata(...)` to do the alignment and clonotype extraction:
- intersects receptor-positive cells (`gex ∩ airr`)
- resolves clonotype (default `clonotype_key="auto"`)
- wires one covariate column from `obs`
- ensures the counts layer exists

If you want a composite covariate (for example tissue × timepoint), build that column upstream first and pass the single combined column name.


In [ ]:
adata = tcri.pp.from_mudata(
    mdata,
    clonotype_key="auto",
    covariate_cols="tissue",
)

adata


In [ ]:
# optional EDA check
pd.crosstab(adata.obs["tissue"], adata.obs["timepoint"])


## Register with tcri

`tcri` requires a phenotype column. If you already have one (e.g. cell-state annotation), use it.
For this loading demo, we fall back to `tissue` as a placeholder to make `setup_anndata` runnable.

In [ ]:
phenotype_key = "phenotype"
if phenotype_key not in adata.obs:
    adata.obs[phenotype_key] = adata.obs["tissue"].astype(str).astype("category")

tcri.ml.TCRIModel.setup_anndata(
    adata,
    layer="counts",
    clonotype_key="clone_id",
    phenotype_key=phenotype_key,
    covariate_key="tissue_site",  # tissue x site
    batch_key="patient",          # requested batch identifier
    replicate="patient",
)

adata.obs[["clone_id", phenotype_key, "tissue_site", "patient"]].head()

You can now continue with standard `tcri` steps:

```python
model = tcri.ml.TCRIModel(adata)
model.train(max_epochs=200, batch_size=512)
model.to_anndata(adata)
tcri.null.all(model, adata)
```
